# Testing the Inputs on Small Examples

Before trust the real data, we test our steps on two tiny made-up homes (A and B) where we already know the right answers. If the notebook gets these right, we can be more confident it handles the real homes correctly.

## Project folder

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path("../..").resolve()
PROJECT_ROOT

WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction')

## Making a test home (A)

Home A uses 0.0, 0.1, 0.2 ... kWh, so it's easy to tell which hour a value came from.

In [ ]:
import pandas as pd
import numpy as np
start = pd.Timestamp("2013-01-01", tz="UTC")
times = pd.date_range(start, periods=50, freq="h")
home_a = pd.DataFrame({"LCLid": "A", "hour": times, "kwh": np.arange(50) / 10})
home_a.loc[26, "kwh"] = np.nan
home_a.loc[10, "kwh"] = 0.0
home_a.head()

,LCLid,hour,kwh
0,A,2013-01-01 00:00:00+00:00,0.0
1,A,2013-01-01 01:00:00+00:00,0.1
2,A,2013-01-01 02:00:00+00:00,0.2
3,A,2013-01-01 03:00:00+00:00,0.3
4,A,2013-01-01 04:00:00+00:00,0.4


## Removing hour 3 and adding a second home (B)

Taking out one hour lets us check that gaps are handled properly.

In [ ]:
home_a = home_a.drop(index=3)
home_b = pd.DataFrame({"LCLid": "B", "hour": times[:30], "kwh": 100.0 + np.arange(30)})
hourly = pd.concat([home_a, home_b], ignore_index=True)
hourly.head()

,LCLid,hour,kwh
0,A,2013-01-01 00:00:00+00:00,0.0
1,A,2013-01-01 01:00:00+00:00,0.1
2,A,2013-01-01 02:00:00+00:00,0.2
3,A,2013-01-01 04:00:00+00:00,0.4
4,A,2013-01-01 05:00:00+00:00,0.5


## Listing every hour

In [ ]:
hours = pd.date_range(hourly["hour"].min(), hourly["hour"].max(), freq="h")
hours[:5]

DatetimeIndex(['2013-01-01 00:00:00+00:00', '2013-01-01 01:00:00+00:00',
               '2013-01-01 02:00:00+00:00', '2013-01-01 03:00:00+00:00',
               '2013-01-01 04:00:00+00:00'],
              dtype='datetime64[us, UTC]', freq='h')

## One row for every home and every hour

In [ ]:
homes = sorted(hourly["LCLid"].unique())
grid = pd.MultiIndex.from_product([homes, hours], names=["LCLid", "target_hour"])
len(grid)

100

## Lining up the readings by home and hour

In [ ]:
values = hourly.set_index(["LCLid", "hour"])["kwh"]
values.index.names = ["LCLid", "target_hour"]
values.head()

LCLid  target_hour              
A      2013-01-01 00:00:00+00:00    0.0
       2013-01-01 01:00:00+00:00    0.1
       2013-01-01 02:00:00+00:00    0.2
       2013-01-01 04:00:00+00:00    0.4
       2013-01-01 05:00:00+00:00    0.5
Name: kwh, dtype: float64

## Filling in the hours with no reading

In [ ]:
values = values.reindex(grid)
values.head()

LCLid  target_hour              
A      2013-01-01 00:00:00+00:00    0.0
       2013-01-01 01:00:00+00:00    0.1
       2013-01-01 02:00:00+00:00    0.2
       2013-01-01 03:00:00+00:00    NaN
       2013-01-01 04:00:00+00:00    0.4
Name: kwh, dtype: float64

## Adding the value we want to predict

In [ ]:
all_rows = values.reset_index(name="target_kwh")
all_rows.head()

,LCLid,target_hour,target_kwh
0,A,2013-01-01 00:00:00+00:00,0.0
1,A,2013-01-01 01:00:00+00:00,0.1
2,A,2013-01-01 02:00:00+00:00,0.2
3,A,2013-01-01 03:00:00+00:00,NaN
4,A,2013-01-01 04:00:00+00:00,0.4


## Find the previous hour

In [ ]:
previous_time = all_rows["target_hour"] - pd.Timedelta(hours=1)
keys = pd.MultiIndex.from_arrays([all_rows["LCLid"], previous_time])
previous_values = values.reindex(keys)
all_rows["lag_1h_kwh"] = previous_values.values
all_rows.head()

,LCLid,target_hour,target_kwh,lag_1h_kwh
0,A,2013-01-01 00:00:00+00:00,0.0,NaN
1,A,2013-01-01 01:00:00+00:00,0.1,0.0
2,A,2013-01-01 02:00:00+00:00,0.2,0.1
3,A,2013-01-01 03:00:00+00:00,NaN,0.2
4,A,2013-01-01 04:00:00+00:00,0.4,NaN


## Find the same hour yesterday

In [ ]:
yesterday_time = all_rows["target_hour"] - pd.Timedelta(hours=24)
keys = pd.MultiIndex.from_arrays([all_rows["LCLid"], yesterday_time])
yesterday_values = values.reindex(keys)
all_rows["lag_24h_kwh"] = yesterday_values.values
all_rows.iloc[22:27]

,LCLid,target_hour,target_kwh,lag_1h_kwh,lag_24h_kwh
22,A,2013-01-01 22:00:00+00:00,2.2,2.1,NaN
23,A,2013-01-01 23:00:00+00:00,2.3,2.2,NaN
24,A,2013-01-02 00:00:00+00:00,2.4,2.3,0.0
25,A,2013-01-02 01:00:00+00:00,2.5,2.4,0.1
26,A,2013-01-02 02:00:00+00:00,NaN,2.5,0.2


## Adding the hour of day and weekday

In [ ]:
all_rows["hour"] = all_rows["target_hour"].dt.hour
all_rows["weekday"] = all_rows["target_hour"].dt.dayofweek
all_rows.iloc[22:27]

C:\Users\Mystery\AppData\Local\Temp\ipykernel_66024\3353378970.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  all_rows["weekday"] = all_rows["target_hour"].dt.dayofweek


,LCLid,target_hour,target_kwh,lag_1h_kwh,lag_24h_kwh,hour,weekday
22,A,2013-01-01 22:00:00+00:00,2.2,2.1,NaN,22,1
23,A,2013-01-01 23:00:00+00:00,2.3,2.2,NaN,23,1
24,A,2013-01-02 00:00:00+00:00,2.4,2.3,0.0,0,2
25,A,2013-01-02 01:00:00+00:00,2.5,2.4,0.1,1,2
26,A,2013-01-02 02:00:00+00:00,NaN,2.5,0.2,2,2


## Find rows with missing values

In [ ]:
all_rows["row_status"] = "kept"
all_rows.loc[all_rows["lag_24h_kwh"].isna(), "row_status"] = "missing_lag_24h"
all_rows.loc[all_rows["lag_1h_kwh"].isna(), "row_status"] = "missing_lag_1h"
all_rows.loc[all_rows["target_kwh"].isna(), "row_status"] = "missing_target"
all_rows["row_status"].value_counts()

row_status
missing_lag_24h    44
kept               30
missing_target     22
missing_lag_1h      4
Name: count, dtype: int64

## Keeping the rows we can use

In [ ]:
features = all_rows[all_rows["row_status"] == "kept"]
features = features.drop(columns="row_status")
features.head()

,LCLid,target_hour,target_kwh,lag_1h_kwh,lag_24h_kwh,hour,weekday
24,A,2013-01-02 00:00:00+00:00,2.4,2.3,0.0,0,2
25,A,2013-01-02 01:00:00+00:00,2.5,2.4,0.1,1,2
28,A,2013-01-02 04:00:00+00:00,2.8,2.7,0.4,4,2
29,A,2013-01-02 05:00:00+00:00,2.9,2.8,0.5,5,2
30,A,2013-01-02 06:00:00+00:00,3.0,2.9,0.6,6,2


## How many rows did we keep?

In [ ]:
len(features)

30

## Counting the gaps for each home

In [ ]:
counts = pd.crosstab(all_rows["LCLid"], all_rows["row_status"])
counts = counts.reindex(columns=["kept", "missing_target",
    "missing_lag_1h", "missing_lag_24h"], fill_value=0)
counts.columns = ["rows_kept", "excluded_missing_target",
    "excluded_missing_lag_1h", "excluded_missing_lag_24h"]
counts["grid_rows"] = all_rows.groupby("LCLid").size()
counts

,rows_kept,excluded_missing_target,excluded_missing_lag_1h,excluded_missing_lag_24h,grid_rows
LCLid,,,,,
A,24,2,3,21,50
B,6,20,1,23,50


## Breaking the gaps down by cause

In [ ]:
for field, name in [("target_kwh", "any_target_missing"),
                    ("lag_1h_kwh", "any_lag_1h_missing"), ("lag_24h_kwh", "any_lag_24h_missing")]:
    counts[name] = all_rows[field].isna().groupby(all_rows["LCLid"]).sum()
counts

,rows_kept,excluded_missing_target,excluded_missing_lag_1h,excluded_missing_lag_24h,grid_rows,any_target_missing,any_lag_1h_missing,any_lag_24h_missing
LCLid,,,,,,,,
A,24,2,3,21,50,2,3,25
B,6,20,1,23,50,20,20,24


## How many targets do we have on day one?

In [ ]:
first_hour = all_rows.groupby("LCLid")["target_hour"].transform("min")
first_day = all_rows["target_hour"] < first_hour + pd.Timedelta(hours=24)
observed = first_day & all_rows["target_kwh"].notna()
counts["target_observed_in_first_24h"] = observed.groupby(all_rows["LCLid"]).sum()
counts

,rows_kept,excluded_missing_target,excluded_missing_lag_1h,excluded_missing_lag_24h,grid_rows,any_target_missing,any_lag_1h_missing,any_lag_24h_missing,target_observed_in_first_24h
LCLid,,,,,,,,,
A,24,2,3,21,50,2,3,25,23
B,6,20,1,23,50,20,20,24,24


## Putting the count table together

In [ ]:
column_order = ["grid_rows", "rows_kept", "excluded_missing_target", "excluded_missing_lag_1h",
                "excluded_missing_lag_24h", "any_target_missing", "any_lag_1h_missing",
                "any_lag_24h_missing", "target_observed_in_first_24h"]
counts = counts[column_order].copy()
counts.loc["TOTAL"] = counts.sum()
counts = counts.reset_index()
counts

,LCLid,grid_rows,rows_kept,excluded_missing_target,excluded_missing_lag_1h,excluded_missing_lag_24h,any_target_missing,any_lag_1h_missing,any_lag_24h_missing,target_observed_in_first_24h
0,A,50,24,2,3,21,2,3,25,23
1,B,50,6,20,1,23,20,20,24,24
2,TOTAL,100,30,22,4,44,22,23,49,47


## Making the examples easy to look up

In [ ]:
examples = all_rows.copy()
examples["elapsed_hour"] = ((examples["target_hour"] - start) / pd.Timedelta(hours=1)).astype(int)
a = examples[examples["LCLid"] == "A"].set_index("elapsed_hour")
b = examples[examples["LCLid"] == "B"].set_index("elapsed_hour")
a.head()

,LCLid,target_hour,target_kwh,lag_1h_kwh,lag_24h_kwh,hour,weekday,row_status
elapsed_hour,,,,,,,,
0,A,2013-01-01 00:00:00+00:00,0.0,NaN,NaN,0,1,missing_lag_1h
1,A,2013-01-01 01:00:00+00:00,0.1,0.0,NaN,1,1,missing_lag_24h
2,A,2013-01-01 02:00:00+00:00,0.2,0.1,NaN,2,1,missing_lag_24h
3,A,2013-01-01 03:00:00+00:00,NaN,0.2,NaN,3,1,missing_target
4,A,2013-01-01 04:00:00+00:00,0.4,NaN,NaN,4,1,missing_lag_1h


## Test 1: a normal row gives the expected values

In [ ]:
fields = ["target_kwh", "lag_1h_kwh", "lag_24h_kwh", "hour", "weekday"]
normal = pd.DataFrame({"actual": a.loc[30, fields], "expected": [3.0, 2.9, 0.6, 6, 2]})
normal["matches"] = np.isclose(normal["actual"].astype(float), normal["expected"])
normal

,actual,expected,matches
target_kwh,3.0,3.0,True
lag_1h_kwh,2.9,2.9,True
lag_24h_kwh,0.6,0.6,True
hour,6.0,6.0,True
weekday,2.0,2.0,True


## Test 2: a missing hour doesn't get skipped over

If hour 3 is missing, hour 4's "previous hour" should be empty, not hour 2's value.

In [ ]:
gap_examples = a.loc[[4, 5, 27, 28], ["lag_1h_kwh", "lag_24h_kwh", "row_status"]]
gap_examples

,lag_1h_kwh,lag_24h_kwh,row_status
elapsed_hour,,,
4,NaN,NaN,missing_lag_1h
5,0.4,NaN,missing_lag_24h
27,NaN,NaN,missing_lag_1h
28,2.7,0.4,kept


## Test 3: a missing target stays missing

In [ ]:
a.loc[26, ["target_kwh", "row_status"]]

target_kwh               NaN
row_status    missing_target
Name: 26, dtype: object

## Making sure the kept rows have no gaps

This should print False, meaning nothing is missing.

In [ ]:
features.isna().any().any()

np.False_

## Test 4: a missing input stays missing

In [ ]:
a.loc[27, ["lag_1h_kwh", "row_status"]]

lag_1h_kwh               NaN
row_status    missing_lag_1h
Name: 27, dtype: object

## Test 5: real zero isn't treated as missing

Zero means the home used no power. That's a real reading, so the row should be kept.

In [ ]:
zero_values = pd.Series({"target_at_10": a.loc[10, "target_kwh"],
                         "lag_1h_at_11": a.loc[11, "lag_1h_kwh"],
                         "lag_24h_at_34": a.loc[34, "lag_24h_kwh"]})
zero_values, a.loc[34, "row_status"]

(target_at_10     0.0
 lag_1h_at_11     0.0
 lag_24h_at_34    0.0
 dtype: float64,
 'kept')

## Test 6: each home only uses its own history

In [ ]:
b.loc[[0, 24], ["lag_1h_kwh", "lag_24h_kwh", "row_status"]]

,lag_1h_kwh,lag_24h_kwh,row_status
elapsed_hour,,,
0,NaN,NaN,missing_lag_1h
24,123.0,100.0,kept


## Checking every kept row, home by home

In [ ]:
(b.loc[b["row_status"] == "kept", "lag_1h_kwh"] >= 100).all(), (a.loc[a["row_status"] == "kept",
    "lag_1h_kwh"] < 10).all()

(np.True_, np.True_)

## Test 7: working out the expected values by hand

In [ ]:
kept_a = a[a["row_status"] == "kept"]
i = kept_a.index.to_numpy()
expected = pd.DataFrame(index=kept_a.index)
expected["target_kwh"] = np.where(i == 10, 0, i / 10)
expected["lag_1h_kwh"] = np.where(i - 1 == 10, 0, (i - 1) / 10)
expected["lag_24h_kwh"] = np.where(i - 24 == 10, 0, (i - 24) / 10)
expected.head()

,target_kwh,lag_1h_kwh,lag_24h_kwh
elapsed_hour,,,
24,2.4,2.3,0.0
25,2.5,2.4,0.1
28,2.8,2.7,0.4
29,2.9,2.8,0.5
30,3.0,2.9,0.6


## Comparing every kept row for home A

In [ ]:
history_matches = pd.DataFrame(np.isclose(kept_a[expected.columns], expected), columns=expected.columns)
history_matches.all()

target_kwh     True
lag_1h_kwh     True
lag_24h_kwh    True
dtype: bool

## Test 8: midnight and the weekday are correct

In [ ]:
a.loc[48, ["target_hour", "hour", "weekday", "lag_1h_kwh"]]

target_hour    2013-01-03 00:00:00+00:00
hour                                   0
weekday                                3
lag_1h_kwh                           4.7
Name: 48, dtype: object

## Test 9: the counts add up

In [ ]:
counts[["LCLid", "grid_rows", "rows_kept", "any_target_missing"]]

,LCLid,grid_rows,rows_kept,any_target_missing
0,A,50,24,2
1,B,50,6,20
2,TOTAL,100,30,22


## Comparing the row counts

In [ ]:
count_fields = ["rows_kept", "excluded_missing_target",
    "excluded_missing_lag_1h", "excluded_missing_lag_24h"]
count_balance = counts[count_fields].sum(axis=1) == counts["grid_rows"]
count_balance

0    True
1    True
2    True
dtype: bool

## Test 10: catching duplicate rows

In [ ]:
duplicate_input = pd.concat([hourly, hourly.iloc[[5]]], ignore_index=True)
duplicate_rows = duplicate_input[duplicate_input.duplicated(["LCLid", "hour"], keep=False)]
duplicate_rows

,LCLid,hour,kwh
5,A,2013-01-01 06:00:00+00:00,0.6
79,A,2013-01-01 06:00:00+00:00,0.6


## Test 11: catching a timestamp that isn't on the hour

In [ ]:
off_hour_input = hourly.copy()
off_hour_input.loc[0, "hour"] += pd.Timedelta(minutes=30)
off_hour_rows = off_hour_input[off_hour_input["hour"] != off_hour_input["hour"].dt.floor("h")]
off_hour_rows

,LCLid,hour,kwh
0,A,2013-01-01 00:30:00+00:00,0.0


## Test 12: saving the results

In [ ]:
from pathlib import Path
output_dir = PROJECT_ROOT / 'data/processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/processed')

## Writing both files

In [ ]:
feature_path = output_dir / "input_examples.csv"
counts_path = output_dir / "input_example_counts.csv"
features.to_csv(feature_path, index=False)
counts.to_csv(counts_path, index=False)
feature_path, counts_path

(WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/processed/input_examples.csv'),
 WindowsPath('G:/Other computers/PC Sharing/University/26-27/Data/Project/household-energy-prediction/data/processed/input_example_counts.csv'))

## Reading the saved file back

In [ ]:
saved_features = pd.read_csv(feature_path)
saved_features["target_hour"] = pd.to_datetime(saved_features["target_hour"], utc=True)
saved_features.head()

,LCLid,target_hour,target_kwh,lag_1h_kwh,lag_24h_kwh,hour,weekday
0,A,2013-01-02 00:00:00+00:00,2.4,2.3,0.0,0,2
1,A,2013-01-02 01:00:00+00:00,2.5,2.4,0.1,1,2
2,A,2013-01-02 04:00:00+00:00,2.8,2.7,0.4,4,2
3,A,2013-01-02 05:00:00+00:00,2.9,2.8,0.5,5,2
4,A,2013-01-02 06:00:00+00:00,3.0,2.9,0.6,6,2


## Checking the saved files match

The False here is a good sign: it means the saved file has no missing values.

In [ ]:
saved_counts = pd.read_csv(counts_path)
same_keys = saved_features[["LCLid", "target_hour"]].equals(features[["LCLid",
    "target_hour"]].reset_index(drop=True))
same_values = np.allclose(saved_features[fields], features[fields], rtol=0, atol=1e-12)
list(saved_features.columns), same_keys, same_values, saved_features.isna().any().any(), saved_counts.equals(counts)

(['LCLid',
  'target_hour',
  'target_kwh',
  'lag_1h_kwh',
  'lag_24h_kwh',
  'hour',
  'weekday'],
 True,
 True,
 np.False_,
 True)